# Módulo 2 (continuação) — Rotulagem dos fluxos NFStream a partir dos PCAPs da CICIDS2017
**Objetivo:** reconstruir a coluna `Label` para os fluxos extraídos com NFStream a partir dos PCAPs originais da CICIDS2017, usando IP de origem/destino e o timestamp `bidirectional_first_seen_ms`, com base nas informações publicadas em https://www.unb.ca/cic/datasets/ids-2017.html.
**Autor:** Marcel Teixeira Chiarelo
**Data:** julho/2026


## 1. Contexto

A rotulagem é feita **sem** depender do CSV original da CICIDS2017 (gerado pelo CICFlowMeter): uso apenas as informações públicas do site do CIC — IP de atacante, IP de vítima e o intervalo de horário de cada ataque — combinadas com `src_ip`, `dst_ip` e `bidirectional_first_seen_ms` (timestamp de início do fluxo) de cada fluxo extraído pelo NFStream.

As janelas de ataque (data, hora de início, hora de fim, atacante, vítima) foram copiadas da própria página do CIC e estão centralizadas no arquivo `rotulagem_cicids2017.py`, junto desta notebook. Os horários são convertidos para o fuso `America/Moncton` (horário local do laboratório da UNB).

**Atenção ao NAT do firewall:** o site do CIC documenta que o tráfego do atacante (Kali, IP público `205.174.165.73`) passa por NAT antes de chegar na rede da vítima: `205.174.165.73 → 205.174.165.80 → 172.16.0.1 → vítima`. Como os PCAPs foram capturados dentro da rede interna (os IPs dos fluxos são da faixa `192.168.10.x`), **o atacante aparece como `172.16.0.1`, não como o IP público**. Por isso `rotulagem_cicids2017.py` aceita os dois IPs como candidatos a atacante.

**Limitação conhecida (documentar no relatório):** Engelen, Rimmer & Joosen (2021), em *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study*, mostram que as janelas de horário oficiais não são perfeitamente exatas. Por isso a Seção 4 compara a distribuição de classes obtida aqui com a do CSV original, para ter uma ideia da qualidade da rotulagem.


> **Notas de implementação:**
> 1. **Portscan interno da Infiltration (quinta, 15:04–15:45):** essa etapa não tem um IP de vítima único, então uso `"*_local"`: qualquer cliente da rede `192.168.10.x`, excluindo os servidores de infraestrutura (DNS/LDAP, web, e-mail e o gateway NAT). Um coringa total (`"*"`) rotulava como ataque todo o tráfego normal do host infectado.
> 2. **Comparação com o CSV original (Seção 4):** alguns rótulos do CSV oficial chegam com o traço corrompido (`Web Attack � XSS`). Como o caractere já está perdido no próprio arquivo, comparo os rótulos por uma chave normalizada (`normalizar_label`) em vez da string exata.


In [1]:
import glob
import pandas as pd
from rotulagem_cicids2017 import label_dataframe, ATTACK_WINDOWS, ler_csv_original_cicids2017, normalizar_label

pd.set_option("display.max_columns", None)


## 2. Carregar o CSV gerado pelo NFStream

Uso o CSV de quinta-feira gerado por `extrair_nfstream_pcaps.py` (NFStream rodando sobre o `Thursday-WorkingHours.pcap` inteiro), salvo nesta mesma pasta `modulo2/` como `captura_nfstream_thursday.csv`. Os outros dias (terça, quarta e sexta) são tratados na notebook `modulo2_comparacao_semana.ipynb`.


In [2]:
NFSTREAM_CSVS = ["captura_nfstream_thursday.csv"]
print(f"{len(NFSTREAM_CSVS)} arquivo(s):")
for f in NFSTREAM_CSVS:
    print(" -", f)


1 arquivo(s):
 - captura_nfstream_thursday.csv


In [3]:
dfs = []
for f in NFSTREAM_CSVS:
    d = pd.read_csv(f)
    d["__origem_arquivo"] = f
    dfs.append(d)

df_nfstream = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Total de fluxos carregados: {len(df_nfstream)}")
df_nfstream.head()


Total de fluxos carregados: 343134


,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,protocol,ip_version,vlan_id,tunnel_id,bidirectional_first_seen_ms,bidirectional_last_seen_ms,bidirectional_duration_ms,bidirectional_packets,bidirectional_bytes,src2dst_first_seen_ms,src2dst_last_seen_ms,src2dst_duration_ms,src2dst_packets,src2dst_bytes,dst2src_first_seen_ms,dst2src_last_seen_ms,dst2src_duration_ms,dst2src_packets,dst2src_bytes,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,__origem_arquivo
0,0,0,fe80::6d07:ea13:6f73:ed41,b8:ac:6f:36:08:f5,b8:ac:6f,0,ff02::16,33:33:00:00:00:16,33:33:00,0,58,6,0,0,1499342465192,1499342468769,3577,43,3870,1499342465192,1499342468769,3577,43,3870,0,0,0,0,0,ICMPV6,Network,0,6,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday.csv
1,1,0,192.168.10.8,b8:ac:6f:36:08:f5,b8:ac:6f,0,224.0.0.22,01:00:5e:00:00:16,01:00:5e,0,2,4,0,0,1499342468690,1499342468767,77,2,120,1499342468690,1499342468767,77,2,120,0,0,0,0,0,IGMP,Network,0,6,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday.csv
2,2,0,192.168.10.19,00:23:ae:9b:ad:b3,00:23:ae,123,167.114.204.238,00:c1:b1:14:eb:31,00:c1:b1,123,17,4,0,0,1499342458269,1499342462285,4016,6,540,1499342458269,1499342462269,4000,3,270,1499342458285,1499342462285,4000,3,270,NTP,System,0,6,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday.csv
3,3,0,192.168.10.19,00:23:ae:9b:ad:b3,00:23:ae,123,72.38.129.202,00:c1:b1:14:eb:31,00:c1:b1,123,17,4,0,0,1499342450269,1499342469303,19034,6,540,1499342450269,1499342469269,19000,3,270,1499342450301,1499342469303,19002,3,270,NTP,System,0,6,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday.csv
4,4,0,192.168.10.19,00:23:ae:9b:ad:b3,00:23:ae,138,192.168.10.255,ff:ff:ff:ff:ff:ff,ff:ff:ff,138,17,4,0,0,1499342438168,1499342513456,75288,21,5817,1499342438168,1499342513456,75288,21,5817,0,0,0,0,0,NetBIOS.SMBv1,System,0,6,ubuntu14-32,NaN,NaN,NaN,NaN,captura_nfstream_thursday.csv


## 3. Aplicar a rotulagem

Cada fluxo é comparado contra as janelas de ataque (`ATTACK_WINDOWS`, em `rotulagem_cicids2017.py`): se `src_ip`/`dst_ip` corresponderem ao par atacante/vítima da janela (considerando o IP pós-NAT `172.16.0.1`) **e** `bidirectional_first_seen_ms` cair dentro do intervalo de horário, o fluxo recebe o rótulo do ataque; caso contrário, recebe `BENIGN`.


In [4]:
df_rotulado = label_dataframe(
    df_nfstream,
    src_col="src_ip",
    dst_col="dst_ip",
    time_col="bidirectional_first_seen_ms",
)

df_rotulado["Label"].value_counts()


Label
BENIGN                        285981
Infiltration                   55134
Web Attack - Brute Force        1349
Web Attack - XSS                 661
Web Attack - Sql Injection         9
Name: count, dtype: int64

In [5]:
df_rotulado.to_csv("nfstream_cicids2017_rotulado.csv", index=False)
print("Arquivo salvo: nfstream_cicids2017_rotulado.csv")
print(f"Dimensões: {df_rotulado.shape}")


Arquivo salvo: nfstream_cicids2017_rotulado.csv
Dimensões: (343134, 40)


## 4. Checagem de sanidade (comparação rápida com o CSV original)

Antes de seguir para a análise estatística completa (próxima notebook), vale comparar rapidamente a contagem de classes obtida aqui com a contagem dos CSVs originais da CICIDS2017 para quinta-feira. Como o NFStream processou o **dia inteiro**, concatenamos os dois CSVs oficiais de quinta (`Morning-WebAttacks` + `Afternoon-Infilteration`) para a comparação ficar justa (dia completo dos dois lados). Diferenças de alguns % são esperadas (ver limitação na Seção 1); diferenças muito grandes (ex.: 10x) indicam problema de fuso horário ou de par de IPs.

Observação: o CSV original que vocês baixaram é a versão "reduzida" da CICIDS2017 (sem colunas de IP/porta/timestamp), então essa checagem compara apenas a **contagem por classe**, não faz cruzamento fluxo a fluxo.


In [6]:
ORIGINAL_CSVS = sorted(glob.glob("dados/Thursday-*.csv"))
print("CSVs originais encontrados:", ORIGINAL_CSVS)

if ORIGINAL_CSVS:
    orig = ler_csv_original_cicids2017(ORIGINAL_CSVS)
    print("Rotulos unicos no CSV original:", sorted(orig["Label"].unique()))

    # Alguns rotulos do CSV original ja vem com o traco corrompido (U+FFFD),
    # entao comparo por uma chave normalizada (normalizar_label) e nao pela
    # string exata. Os nomes exibidos continuam sendo os originais.
    contagem_nosso = df_rotulado["Label"].value_counts()
    contagem_orig = orig["Label"].value_counts()

    norm_para_nome = {}
    for label in list(contagem_nosso.index) + list(contagem_orig.index):
        norm_para_nome.setdefault(normalizar_label(label), label)

    comparacao = pd.DataFrame({
        "NFStream (nosso)": contagem_nosso.groupby(normalizar_label).sum(),
        "CICIDS2017 (original, dia inteiro)": contagem_orig.groupby(normalizar_label).sum(),
    }).fillna(0).astype(int)
    comparacao.index = comparacao.index.map(norm_para_nome)
    comparacao.index.name = "Label"
    display(comparacao)
else:
    print("Nenhum CSV original encontrado em dados/Thursday-*.csv - ajuste o caminho para rodar a checagem.")


CSVs originais encontrados: ['dados/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'dados/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv']
[ler_csv_original_cicids2017] dados/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: lido com encoding='utf-8'
[ler_csv_original_cicids2017] dados/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: lido com encoding='utf-8'
Rotulos unicos no CSV original: ['BENIGN', 'Infiltration', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']


,NFStream (nosso),"CICIDS2017 (original, dia inteiro)"
Label,,
BENIGN,285981,456752
Infiltration,55134,36
Web Attack - Brute Force,1349,1507
Web Attack - Sql Injection,9,21
Web Attack - XSS,661,652


## Reflexão

- **Fluxos rotulados:** o NFStream gerou 343.134 fluxos para quinta-feira. Desses, 285.981 (83,34%) ficaram como BENIGN e 57.153 (16,66%) como ataque: 55.134 Infiltration, 1.349 Web Attack - Brute Force, 661 Web Attack - XSS e 9 Web Attack - Sql Injection.

- **Distribuição em relação ao CSV original:** o CSV oficial de quinta tem 458.968 fluxos, com 99,52% BENIGN e 0,48% ataque (2.216 fluxos). A minha rotulagem classifica proporcionalmente muito mais tráfego como ataque, e quase tudo isso é a classe Infiltration.

- **Web Attack:** ficaram próximos do oficial: Brute Force 1.349 contra 1.507, XSS 661 contra 652 e Sql Injection 9 contra 21. Essas janelas têm um par de IP específico (atacante e servidor web), e o NFStream gera menos fluxos que o CICFlowMeter, o que explica contagens um pouco menores. A diferença de 12 fluxos no Sql Injection pesa pouco em valor absoluto, mas é grande em proporção por ser uma classe muito pequena.

- **Infiltration:** 55.134 contra 36 no oficial, a divergência mais forte. A segunda etapa do ataque é um portscan interno da máquina infectada (`192.168.10.8`) contra a rede, e o site do CIC não informa um IP de vítima único para ela. Usei o critério `*_local` (qualquer cliente de `192.168.10.x`, excluindo servidores de infraestrutura), que é muito mais amplo do que os poucos pares de IP rotulados no CSV oficial. Ainda não decompus o total por janela, então atribuir a maior parte à janela de portscan é uma hipótese a confirmar.

- **Volume de fluxos:** o NFStream gerou 74,8% dos fluxos do oficial (343.134 contra 458.968). Isso aparece em todos os dias e é esperado, porque as duas ferramentas fecham fluxos de forma diferente.

- **Limitação na comparação:** alguns rótulos do CSV oficial já vêm com o traço corrompido (`Web Attack � XSS`), então comparo por uma chave normalizada (`normalizar_label`) e não pela string exata.

- Este arquivo (`nfstream_cicids2017_rotulado.csv`) é a entrada da notebook `modulo2_analise_comparativa.ipynb`.